<a href="https://colab.research.google.com/github/abdullahawan0043-glitch/Flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import os, subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/abdullahawan0043-glitch/Flyrank-machine-learning-internship"
REPO_DIR = "Flyrank-machine-learning-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

Loaded 30000 rows


In [ ]:
"""
Rule idea in plain words:
A page is worth reviewing first if it hasn't been updated in a long time
AND it still gets meaningful search visibility - a stale page that people
can still find is a wasted opportunity. I'll also check whether pages
with good position but low CTR (the CTR-fix signal from the session) are
common enough in my data to matter.
"""

# Signal 1: Staleness (behind the refresh flags from the session)
df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                              bins=[-1, 90, 180, 365, 99999],
                              labels=["<90d", "90-180d", "180-365d", "365d+"])
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

stale_table = df.groupby("stale_bucket", observed=True).agg(
    n=("content_id", "count"),
    decline_rate=("is_declining", "mean")
).reset_index()
print("Signal 1: Staleness vs decline rate")
print(stale_table)

base_rate = df["is_declining"].mean()
print(f"\nOverall decline rate (base rate): {base_rate:.3f}")

most_stale_rate = stale_table.iloc[-1]["decline_rate"]
if most_stale_rate > base_rate * 1.2:
    verdict1 = "CONFIRMED"
elif most_stale_rate < base_rate * 0.8:
    verdict1 = "OPPOSITE"
else:
    verdict1 = "MIXED"
print(f"VERDICT: {verdict1} - staleness {'is' if verdict1=='CONFIRMED' else 'is not clearly'} linked to decline")

Signal 1: Staleness vs decline rate
  stale_bucket      n  decline_rate
0         <90d  20655      0.512031
1      90-180d   9171      0.611057
2     180-365d    169      0.467456
3        365d+      5      0.600000

Overall decline rate (base rate): 0.542
VERDICT: MIXED - staleness is not clearly linked to decline


In [ ]:
# Signal 2: CTR vs Position (behind the CTR-fix logic)
df["position_bucket"] = pd.cut(df["avg_position"],
                                 bins=[-1, 0, 3, 10, 20, 100],
                                 labels=["no_data", "top3", "4-10", "11-20", "20+"])

ctr_table = df[df["avg_position"] > 0].groupby("position_bucket", observed=True).agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()
print("Signal 2: Position vs average CTR")
print(ctr_table)

top_ctr = ctr_table[ctr_table["position_bucket"] == "top3"]["avg_ctr"].values
low_pos_ctr = ctr_table[ctr_table["position_bucket"] == "11-20"]["avg_ctr"].values

if len(top_ctr) and len(low_pos_ctr) and top_ctr[0] > low_pos_ctr[0]:
    verdict2 = "CONFIRMED"
else:
    verdict2 = "MIXED"
print(f"VERDICT: {verdict2} - better position {'does' if verdict2=='CONFIRMED' else 'does not clearly'} mean higher CTR")

Signal 2: Position vs average CTR
  position_bucket      n   avg_ctr
0            top3   1141  2.714303
1            4-10  11842  0.651045
2           11-20   7273  0.323443
3             20+   8524  0.211705
VERDICT: CONFIRMED - better position does mean higher CTR


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
os.makedirs("work/outputs", exist_ok=True)

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

df["baseline_score"] = stale * visible * df["impressions_90d"]

def reason_code(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_but_visible"
    return "no_flag"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = np.where(df["baseline_score"] > 0, "review_for_refresh", "monitor")

queue = df.sort_values("baseline_score", ascending=False)[
    ["content_id", "client_id", "baseline_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "ctr", "trend_direction"]
]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
print(queue.head(10))

Wrote 30000 rows to work/outputs/baseline_action_score.csv
                 content_id          client_id  baseline_score  \
16751  content_cf56e2e2e282  client_7f2253d7e2           61678   
16514  content_7368877ea310  client_7f2253d7e2           59472   
7021   content_1bfaa38ff26c  client_7f2253d7e2           25715   
21268  content_0a91db491d14  client_7f2253d7e2           13299   
11489  content_5feee3994adb  client_7f2253d7e2            7812   
12045  content_c2d929d83eaa  client_7f2253d7e2            7558   
698    content_b16bd7307b39  client_7f2253d7e2            4590   
5327   content_fe16a55cd13d  client_7f2253d7e2            4556   
26810  content_ecb6215e79fd  client_7f2253d7e2            4429   
20837  content_928af3e22c80  client_7f2253d7e2            1697   

             reason_code              action  days_since_last_update  \
16751  stale_but_visible  review_for_refresh                     194   
16514  stale_but_visible  review_for_refresh                     194  

In [ ]:
# Precision@K evaluation, with base rate for context
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p50 = precision_at_k(df["baseline_score"], df["is_declining"], 50)
print(f"Baseline Precision@50: {p50:.3f}")
print(f"Base rate (random guessing): {df['is_declining'].mean():.3f}")

import json
metrics = {"precision_at_50": float(p50), "base_rate": float(df["is_declining"].mean())}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics to work/outputs/baseline_metrics.json")

Baseline Precision@50: 0.680
Base rate (random guessing): 0.542
Saved metrics to work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top10 = queue.head(10).reset_index(drop=True)
print("TOP 10 REVIEW:\n")
for i, row in top10.iterrows():
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} | reason={row['reason_code']}")
    print(f"   Why it's here: {row['days_since_last_update']} days since update, "
          f"{row['impressions_90d']} impressions (stale + still visible).")
    print(f"   What would make it wrong: if this page's traffic is seasonal and will "
          f"naturally recover without a refresh, or if a related page on the same site "
          f"absorbed its lost traffic (consolidation), this flag would be a false positive.\n")

TOP 10 REVIEW:

1. content_id=content_cf56e2e2e282 | action=review_for_refresh | reason=stale_but_visible
   Why it's here: 194 days since update, 61678 impressions (stale + still visible).
   What would make it wrong: if this page's traffic is seasonal and will naturally recover without a refresh, or if a related page on the same site absorbed its lost traffic (consolidation), this flag would be a false positive.

2. content_id=content_7368877ea310 | action=review_for_refresh | reason=stale_but_visible
   Why it's here: 194 days since update, 59472 impressions (stale + still visible).
   What would make it wrong: if this page's traffic is seasonal and will naturally recover without a refresh, or if a related page on the same site absorbed its lost traffic (consolidation), this flag would be a false positive.

3. content_id=content_1bfaa38ff26c | action=review_for_refresh | reason=stale_but_visible
   Why it's here: 194 days since update, 25715 impressions (stale + still visible).
   W

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
"""
Weakest picks in my top 10: any row where impressions_90d is just barely
above the 500 threshold and days_since_last_update is just barely above
180 - these are borderline cases where a small measurement noise could
flip them in or out of the queue. I'd want a persistence check (is the
decline sustained across multiple recent windows, not just one) before
trusting these borderline rows as strongly as the clearer cases.
"""

"\nWeakest picks in my top 10: any row where impressions_90d is just barely\nabove the 500 threshold and days_since_last_update is just barely above\n180 - these are borderline cases where a small measurement noise could\nflip them in or out of the queue. I'd want a persistence check (is the\ndecline sustained across multiple recent windows, not just one) before\ntrusting these borderline rows as strongly as the clearer cases.\n"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.